# Setup and Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType, StructType, StructField
import folium
import matplotlib.pyplot as plt
import pandas as pd

# Create Spark session

In [2]:
spark = SparkSession.builder\
        .appName("SimulateCasablancaTrips")\
        .getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark

# Load Porto data

In [4]:
porto_df = spark.read.csv("s3a://taasim/raw/porto-trips/train.csv", header=True)
# print(f"Total trips: {porto_df.count()}")
porto_df.printSchema()

root
 |-- TRIP_ID: string (nullable = true)
 |-- CALL_TYPE: string (nullable = true)
 |-- ORIGIN_CALL: string (nullable = true)
 |-- ORIGIN_STAND: string (nullable = true)
 |-- TAXI_ID: string (nullable = true)
 |-- TIMESTAMP: string (nullable = true)
 |-- DAY_TYPE: string (nullable = true)
 |-- MISSING_DATA: string (nullable = true)
 |-- POLYLINE: string (nullable = true)



In [5]:
# Parse POLYLINE (JSON array of [lon, lat]) into an array of doubles
gps_schema = ArrayType(ArrayType(DoubleType()))
porto_parsed = porto_df.withColumn("coords", F.from_json(F.col("POLYLINE"), gps_schema))

In [6]:
# Filter out trips with empty or null coordinates
porto_parsed = porto_parsed.filter(F.size("coords") > 0)

In [ ]:
# Explode each trip into individual GPS points (one row per ping)
exploded_df = porto_parsed.withColumn("point", F.explode("coords"))
exploded_df = exploded_df.withColumn("porto_lon", F.col("point")[0]) \
                         .withColumn("porto_lat", F.col("point")[1])

exploded_df.select("TRIP_ID", "porto_lon", "porto_lat").show(5)

# Define Porto ↔ Casablanca bounding boxes

In [10]:
spark.stop()